#### 4. Add a second target (staging or prod) to your databricks.yml with a different workspace host and run_as service principal, and deploy to it.

```bash
As we are using the free tier, we cannot create another workspace but for this kind of task i will create a new workspace, if i use paid services.

manpreetsingh@Manpreet Assignment9 % databricks bundle init
Using profile: botmanpreet
Welcome to the default Python template for Declarative Automation Bundles!

Answer the following questions to customize your project.
You can always change your configuration in the databricks.yml file later.

Note that https://dbc-38bddc5d-02cf.cloud.databricks.com is used for initialization.
(For information on how to change your profile, see https://docs.databricks.com/dev-tools/cli/profiles.html.)

Unique name for this project [my_project]: ^C
manpreetsingh@Manpreet Assignment9 % databricks bundle init
Using profile: assignment-9
Welcome to the default Python template for Declarative Automation Bundles!

Answer the following questions to customize your project.
You can always change your configuration in the databricks.yml file later.

Note that https://dbc-38bddc5d-02cf.cloud.databricks.com is used for initialization.
(For information on how to change your profile, see https://docs.databricks.com/dev-tools/cli/profiles.html.)

Unique name for this project [my_project]: assigment-9-DABs
Validation failed: invalid value for project_name: "assigment-9-DABs". Name must consist of letters, numbers, and underscores.

Unique name for this project [my_project]: assignment_9_DABs
Include a job that runs a notebook: yes
Include an ETL pipeline: no
Include a sample Python package that builds into a wheel file: no
Use serverless compute: yes
Default catalog for any tables created by this project [workspace]: dev
Use a personal schema for each user working on this project
(this is recommended, your personal schema will be 'dev.botmanpreet2021'): no, I will customize the schema configuration 

✨ Your new project has been created in the 'assignment_9_DABs' directory!

To get started, refer to the project README.md file and the documentation at https://docs.databricks.com/dev-tools/bundles/index.html.
manpreetsingh@Manpreet Assignment9 % databricks auth login --host https://dbc-e63a3b85-204d.cloud.databricks.com/

Databricks profile name [dbc-e63a3b85-204d]: manpreet
Profile manpreet was successfully saved
manpreetsingh@Manpreet Assignment9 % databricks bundle validate
Using profile: assignment-9
Name: Assignment9
Target: dev
Workspace:
  Host: https://dbc-38bddc5d-02cf.cloud.databricks.com
  User: botmanpreet2021@gmail.com
  Path: /Workspace/Users/botmanpreet2021@gmail.com/.bundle/Assignment9/dev

Validation OK!
manpreetsingh@Manpreet Assignment9 % databricks bundle deploy --target dev
Using profile: assignment-9
Uploading bundle files to /Workspace/Users/botmanpreet2021@gmail.com/.bundle/Assignment9/dev/files...
Deploying resources...
Deployment complete!
manpreetsingh@Manpreet Assignment9 % databricks bundle deploy --target prod
Error: prod: no such target. Available targets: dev

manpreetsingh@Manpreet Assignment9 % databricks bundle deploy --target prod
Uploading bundle files to /Workspace/Users/botmanpreet2021@gmail.com/.bundle/Assignment9/prod/files...
Deploying resources...
Deployment complete!
manpreetsingh@Manpreet Assignment9 % 

#### 5. Configure M2M (service principal) authentication for the CLI and use it instead of your personal U2M login for a deploy command.

```bash
manpreetsingh@Manpreet Assignment9 % nano ~/.databrickscfg                                
manpreetsingh@Manpreet Assignment9 % databricks clusters list -p M2M-WORKSPACE-PROFILE         

ID  Name  State
manpreetsingh@Manpreet Assignment9 % databricks current-user me --profile M2M-WORKSPACE-PROFILE
{
  "active": true,
  "displayName": "dabs_project",
  "emails": [
    {
      "primary": true,
      "type": "work",
      "value": "549f08cb-c1a2-4048-9b24-f306a4e0e684"
    }
  ],
  "entitlements": [
    {
      "value": "workspace-access"
    },
    {
      "value": "databricks-sql-access"
    },
    {
      "value": "workspace-consume"
    }
  ],
  "groups": [
    {
      "$ref": "Groups/80882219405445",
      "display": "users",
      "type": "direct",
      "value": "80882219405445"
    }
  ],
  "id": "72776679345588",
  "name": {
    "givenName": "dabs_project"
  },
  "schemas": [
    "urn:ietf:params:scim:schemas:core:2.0:User",
    "urn:ietf:params:scim:schemas:extension:workspace:2.0:User"
  ],
  "userName": "549f08cb-c1a2-4048-9b24-f306a4e0e684"
}
manpreetsingh@Manpreet Assignment9 % databricks bundle deploy --target prod --profile M2M-WORKSPACE-PROFILE
Uploading bundle files to /Workspace/Users/549f08cb-c1a2-4048-9b24-f306a4e0e684/.bundle/Assignment9/prod/files...
Deploying resources...
Deployment complete!

#### 6. Write a GitHub Actions workflow that runs databricks bundle validate on every pull request, without deploying anything.


```yaml
name: Validate Databricks Bundle

on:
  pull_request:
    branches:
      - main
      - develop

jobs:
  validate:
    runs-on: ubuntu-latest
    
    steps:
      - name: Checkout code
        uses: actions/checkout@v3
      
      - name: Setup Databricks CLI
        uses: databricks/setup-cli@main
      
      - name: Validate bundle
        run: databricks bundle validate
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
```

**Key points:**

1. **Trigger**: Runs on every pull request to `main` or `develop` branches
2. **No deployment**: Only runs `databricks bundle validate` which checks the bundle configuration without deploying
3. **Required secrets**: You need to configure these in your GitHub repository:
   - `DATABRICKS_HOST`: Your workspace URL 
   - `DATABRICKS_TOKEN`: A Databricks personal access token or service principal token
4. **Setup**: The `databricks/setup-cli@main` action installs the Databricks CLI
